In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/src/small_bench/checkpoints/pre_cell_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

# Helper to rename columns if needed
def col_name_corrections(df, correction_pair):
    if set(df.columns).intersection(correction_pair.keys()):
        df.rename(columns=correction_pair, inplace=True)
    return df

# Collect GPU-backed DataFrames in lists to avoid repeated concat
mobile_list = []
broadband_list = []

for dirpath, _, filenames in os.walk(
    '/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input'
):
    for filename in filenames:
        # Read into GPU-backed DataFrame (patched pd.read_csv)
        df = pd.read_csv(
            os.path.join(dirpath, filename),
            thousands=','
        ).convert_dtypes()
        # Apply any column-name corrections
        df = col_name_corrections(df, {'Number of Record': 'Number of Records'})
        # Extract Year and Quarter from filename
        year = np.int64(re.search(r'year_(\d+)_quarter', filename).group(1))
        quarter = np.int64(re.search(r'quarter_(\d+)\.csv', filename).group(1))
        df['Year'] = year
        df['Quarter'] = quarter
        # Append to corresponding list
        if 'mobile' in filename:
            mobile_list.append(df)
        else:
            broadband_list.append(df)

# Single concat per category (GPU operation)
Mobile_df = pd.concat(mobile_list, ignore_index=True)
Broadband_df = pd.concat(broadband_list, ignore_index=True)

# Ensure Year/Quarter are int64 and sort (all on GPU)
Mobile_df = Mobile_df.astype({'Year': np.int64, 'Quarter': np.int64})
Broadband_df = Broadband_df.astype({'Year': np.int64, 'Quarter': np.int64})
Mobile_df = Mobile_df.sort_values(by=['Year', 'Quarter'], ascending=[True, True])
Broadband_df = Broadband_df.sort_values(by=['Year', 'Quarter'], ascending=[True, True])

print(Broadband_df.shape)
print(Mobile_df.shape)
